# ToxValDB Curation and Preprocessing Pipeline
This notebook serves as the complete initial data curation and preparation step for ToxValDB:
1. Maps species and study details to expected OECD test guidelines.
2. Removes redundant metadata and original columns.
3. Normalizes and tokenizes the `TOXICOLOGICAL_EFFECT` free-text field.
4. Filters for mammalian experimental records.
5. Identifies biological outliers (study duration exceeding species lifespan) and physically/biologically inconsistent numeric values.
6. Filters invalid/suspicious records and exports the cleaned dataset (`ToxValDB_v9.7.0_Cleaned_01.tsv`).


### 1. Import Packages


In [ ]:
import re
import pandas as pd
import numpy as np


### 2. Load Unified ToxValDB Dataset


In [ ]:
df = pd.read_csv('../../00.download_toxvaldb/outputs/ToxValDB_v9.7.0_Unified.tsv', sep='\t', low_memory=False)
print(f"Initial dataset shape: {df.shape}")
df.head()


### 3. Map Expected OECD Test Guidelines


In [ ]:
def get_guideline_matrix_v3(row):
    # --- 0. PRE-PROCESSING ---
    st_type = str(row.get('STUDY_TYPE', '')).lower()
    route = str(row.get('EXPOSURE_ROUTE', '')).lower()
    species = str(row.get('SPECIES_COMMON', '')).lower()
    duration_val = str(row.get('STUDY_DURATION_VALUE', ''))
    duration_class = str(row.get('STUDY_DURATION_CLASS', '')).lower()
    generation = str(row.get('GENERATION', ''))
    toxval_type = str(row.get('TOXVAL_TYPE', '')).lower()
    year = row.get('YEAR', 0)

    # Intentional exclusion
    if 'human' in species:
        return None

    # --- 1. GROUP DEFINITION ---
    is_rabbit = 'rabbit' in species
    is_guinea = 'guinea' in species
    is_mouse  = 'mouse' in species
    is_rat    = 'rat' in species
    is_dog    = 'dog' in species
    is_monkey = 'monkey' in species or 'macaque' in species or 'primate' in species
    is_fish   = any(x in species for x in ['danio', 'fish', 'trout', 'carp', 'minnow'])
    is_bird   = any(x in species for x in ['quail', 'chicken', 'duck', 'mallard', 'bird'])

    # --- 2. ASSIGNMENT LOGIC ---

    # === BLOCK A: ECOTOXICOLOGY ===
    if is_fish:
        return "OECD 203"
    if is_bird:
        return "OECD 205"

    # === BLOCK B: ENDOCRINE ===
    if 'uterotrophic' in st_type:
        return "OECD 440"
    if 'hershberger' in st_type:
        return "OECD 441"

    # === BLOCK C: LOCAL EFFECTS ===
    if is_guinea:
        return "OECD 406"  # Sensitization
    if is_rabbit and ('skin' in st_type or 'dermal' in route or 'irrit' in st_type):
        return "OECD 404"
    if is_rabbit and 'eye' in st_type:
        return "OECD 405"

    # === BLOCK D: ACUTE TOXICITY ===
    if 'acute' in st_type or 'ld50' in toxval_type:
        # Oral (Rat/Mouse)
        if 'oral' in route and (is_rat or is_mouse):
            try:
                y_val = float(str(year).replace(',', '').split('.')[0])
                if y_val >= 2002:
                    return "OECD 423"
                else:
                    return "OECD 401"
            except Exception:
                return "OECD 401"

        # Dermal (Rat/Rabbit)
        if 'dermal' in route and (is_rat or is_rabbit):
            return "OECD 402"
        # Inhalation (Rat)
        if 'inhal' in route and is_rat:
            return "OECD 403"

    # === BLOCK E: REPRO / DEVELOPMENT ===
    if ('F1' in generation or 'F2' in generation or 'P0' in generation) and (is_rat or is_mouse):
        return "OECD 416"
    if ('developmental' in st_type or 'fetus' in generation) and (is_rat or is_rabbit or is_mouse):
        return "OECD 414"
    if 'reproduction' in st_type and (is_rat or is_mouse):
        if 'one' in st_type:
            return "OECD 415"
        return "OECD 421"

    # === BLOCK F: REPEATED DOSE (Subchronic) ===
    if 'subchronic' in st_type or 'repeat' in st_type or 'short-term' in st_type:
        if 'oral' in route and (is_rat or is_mouse):
            if '28' in duration_val or 'short' in duration_class:
                return "OECD 407"
            if '90' in duration_val or 'subchronic' in duration_class:
                return "OECD 408"
            return "OECD 408"

        if 'oral' in route and (is_dog or is_monkey):
            return "OECD 409"

        if 'dermal' in route and (is_rat or is_rabbit or is_guinea):
            return "OECD 410"
        if 'inhal' in route and is_rat:
            return "OECD 412"

    # === BLOCK G: CHRONIC ===
    if 'chronic' in st_type and (is_rat or is_mouse or is_dog or is_monkey):
        return "OECD 452"

    return None

df['MATRIX_GUIDELINE'] = df.apply(get_guideline_matrix_v3, axis=1)
print(f"Guidelines assigned. Non-null guideline counts: {df['MATRIX_GUIDELINE'].notnull().sum()}")


### 4. Remove Redundant Columns and Metadata


In [ ]:
# Drop original copies and internal technical metadata
df = df.drop(columns=[col for col in df.columns if "_ORIGINAL" in col])

df = df.drop(columns=[
    'QC_CATEGORY',
    'QC_STATUS',
    'SOURCE_HASH',
    'SOURCE_URL',
    'SUBSOURCE_URL',
    'source_filename'
], errors='ignore')

print(f"Columns remaining: {df.shape[1]}")


### 5. Normalize and Explode Toxicological Effect Descriptions


In [ ]:
# --- 1. Controlled SPLIT and EXPLODE ---
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].fillna('').astype(str)
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r';', '|', regex=True)

# Deduplicate elements in the list BEFORE exploding
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.split('|').apply(
    lambda x: list(dict.fromkeys([i.strip() for i in x if i.strip()])) if isinstance(x, list) else x
)
df = df.explode('TOXICOLOGICAL_EFFECT').reset_index(drop=True)
print(f"Rows after controlled explode: {df.shape[0]}")

# --- 2. NOISE CLEANING AND NORMALIZATION ---
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.lower().str.strip()

# A. Percentages
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'[~><=+\/-]*\s?\d+[\d\.,]*\s?%', '', regex=True)

# B. Doses and Units
units = r'(mg\/kg(?:\/day)?|ppm|mg\/m3|mg\/l|g\/kg|µg\/kg|mmol\/l|mg\/dl|mg\/kg\s?bw)'
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'(?:at|@)?\s?[~><=]*\s?\d+[\d\.,-]*\s?' + units + r'\b', '', regex=True)

# C. Technical parentheses
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'\([^)]*[\d%][^)]*\)', '', regex=True)

# D. General cleanup of characters and spaces
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Semantic noise filtering
STOP_PHRASES = {
    "terminal sacrifice", "scheduled sacrifice", "study termination", "necropsy", "autopsy"
}
df = df[~df['TOXICOLOGICAL_EFFECT'].isin(STOP_PHRASES)]

# Deduplication after noise filtering
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after noise filtering: {df.shape[0]}")

# --- 3. SEX/SPECIES CLEANUP AND FINAL DEDUPLICATION ---
def remove_in_sex_species(text):
    pattern_in = r'\b(?:in|/)\s*(?:male?s?|female?s?|mice|rats?|rabbits?|dogs?|pigs?|monkeys?)(?:\s+(?:and|or)\s+(?:in\s+|/)?(?:male?s?|female?s?|mice|rats?|rabbits?|dogs?|pigs?|monkeys?))*'
    pattern_parens = r'\(\s*(?:male?s?|female?s?)(?:\s+(?:and|or)\s+(?:male?s?|female?s?))?(?:\s+only)?\s*\)'
    combined = f'({pattern_in}|{pattern_parens})'
    cleaned = re.sub(combined, '', text, flags=re.IGNORECASE | re.VERBOSE)
    return re.sub(r'\s+', ' ', cleaned).strip(' ,')

df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].apply(remove_in_sex_species)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after effect standardization: {df.shape[0]}")


### 6. Filter for Mammalian Studies


In [ ]:
if 'SPECIES_SUPERCATEGORY' in df.columns:
    df = df[df['SPECIES_SUPERCATEGORY'] == 'Mammals'].copy()
    print(f"Rows remaining (Mammals only): {df.shape[0]}")


### 7. Biological Outlier Detection (Study Duration vs Species Lifespan)


In [ ]:
# Mapping of Species to estimated Max Lifespan in YEARS
lifespan_map = {
    'Rat': 4,
    'Mouse': 4,
    'Rabbit': 12,
    'European Rabbit': 12,
    'Human': 120,
    'Dog': 18,
    'Guinea Pig': 8,
    'Monkey': 40,
    'Rhesus Macaque': 40,
    'Cat': 25,
    'Hamster': 4,
    'Pig': 20,
    'Mink': 10,
    'American Mink': 10,
    'Gerbil': 5,
    'Japanese Quail': 3,
    'Chicken': 8,
    'Sheep': 12,
    'Zebra Danio': 5,
    'Northern Bobwhite Quail': 8,
    'Cow Family': 20
}

unit_to_years = {
    'years': 1.0,
    'year': 1.0,
    'months': 1/12,
    'month': 1/12,
    'weeks': 1/52.17,
    'week': 1/52.17,
    'days': 1/365.25,
    'day': 1/365.25,
    'hours': 1/(24*365.25),
    'minutes': 1/(60*24*365.25),
    'generation': 1.0
}

if 'STUDY_DURATION_VALUE' in df.columns and 'STUDY_DURATION_UNITS' in df.columns:
    df['STUDY_DURATION_YEARS'] = df.apply(
        lambda row: row['STUDY_DURATION_VALUE'] * unit_to_years.get(str(row['STUDY_DURATION_UNITS']).lower(), 0)
        if pd.notnull(row['STUDY_DURATION_VALUE']) else np.nan,
        axis=1
    )


### 8. Physical and Biological Consistency Flagging on Numeric Values


In [ ]:
mass_conv_sorted = [
    ('mg/kg-day', 1.0), ('mg/kg/day', 1.0), ('ug/kg-day', 0.001), ('ng/kg-day', 0.000001),
    ('mg/kg bw', 1.0), ('ug/kg bw', 0.001), ('ug/kg', 0.001), ('ng/kg', 0.000001),
    ('mg/kg', 1.0), ('g/kg', 1000.0), ('mg/g', 1000.0), ('ug/mg', 1.0)
]

def classify_toxval_unified(row):
    val = row.get('TOXVAL_NUMERIC')
    unit = str(row.get('TOXVAL_UNITS', '')).lower().strip()
    tox_type = str(row.get('TOXVAL_TYPE', '')).lower().strip()
    if pd.isnull(val):
        return None

    # Exclusions (Specialized Units/Types)
    if '-1' in unit or '^-1' in unit or 'cancer' in tox_type or 'slope' in tox_type or 'mg/m3' in unit:
        return None

    # Physical Errors (> 100% conc)
    if 'ppm' in unit and val > 1000000:
        return 'Error: Physical (> 100% conc)'
    pure_percent = ['%', '% v/v', '% diet', '% w/v', '% weight']
    if unit in pure_percent and val > 100:
        return 'Error: Physical (> 100% conc)'

    # Doses > 1,000,000 mg/kg
    multiplier = None
    for u_str, factor in mass_conv_sorted:
        if u_str in unit:
            multiplier = factor
            break
    if multiplier is not None:
        norm_val = val * multiplier
        if norm_val > 1000000:
            return 'Error: Physical (Dose > Animal Weight)'
        if norm_val > 100000:
            return 'Review: Suspicious (Dose > 100g/kg)'

    # Biological Suspicion (Concentrations > 10%)
    if ('mg/l' in unit or 'ppm' in unit) and val > 100000:
        return 'Review: Suspicious (Concentration > 10%)'

    return None

df['TOXVAL_OUTLIER_FLAG'] = df.apply(classify_toxval_unified, axis=1)
print(f"Suspicious or erroneous ToxVal records flagged: {df['TOXVAL_OUTLIER_FLAG'].notnull().sum()}")


### 9. Apply Exclusion Criteria and Export Cleaned Dataset


In [ ]:
# Define exclusion masks
bio_outliers_mask = (df['STUDY_DURATION_YEARS'] > df['SPECIES_COMMON'].map(lifespan_map))
suspicious_mask = df['TOXVAL_OUTLIER_FLAG'].notnull()

# Keep only experimental records
non_experimental_mask = (df['EXPERIMENTAL_RECORD'] != "experimental")

# Eliminate rows with Human
human_mask = df['SPECIES_COMMON'].str.contains("Human", na=False)

# Eliminate specific non-target species
to_eliminate = ["Sheep", "Rhesus Macaque", "Mink", "Gerbil"]
eliminate_mask = df['SPECIES_COMMON'].isin(to_eliminate)

# Combine all exclusion criteria
to_remove_mask = bio_outliers_mask | suspicious_mask | non_experimental_mask | human_mask | eliminate_mask

print(f"Total rows to remove: {to_remove_mask.sum()}")
print(f" - Biological outliers (Duration > Lifespan): {bio_outliers_mask.sum()}")
print(f" - Suspicious/Error records (TOXVAL_OUTLIER_FLAG): {suspicious_mask.sum()}")
print(f" - Non-experimental records: {non_experimental_mask.sum()}")
print(f" - Human records: {human_mask.sum()}")
print(f" - Specific eliminated species: {eliminate_mask.sum()}")

# Drop flagged rows and remove redundant column
df_cleaned = df[~to_remove_mask].copy()
if 'EXPERIMENTAL_RECORD' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['EXPERIMENTAL_RECORD'])

# Standardize species names
species_map = {
    "Western European House Mouse": "Mouse",
    "House Mouse": "Mouse",
    "Mouse, Rat": "Rat",
    "Guinea Pig, Rat": "Rat"
}
df_cleaned['SPECIES_COMMON'] = df_cleaned['SPECIES_COMMON'].replace(species_map)

# Save final cleaned dataset
output_csv = '../outputs/ToxValDB_v9.7.0_Cleaned_01.tsv'
df_cleaned.to_csv(output_csv, sep='\t', index=False)
print(f"Cleaned dataset saved to: {output_csv}")
print(f"Final dataset shape: {df_cleaned.shape}")
